# Himalaya Sentinel — Risk Model Calibration (v2)

**Notebook**: `ml-notebooks/01_risk_calibration.ipynb`  
**Purpose**: Full feature engineering + model training pipeline for the NER landslide risk engine.  
**Stack**: Python 3.11 + scikit-learn + psycopg2 (offline notebook — does NOT run inside the Supabase/TanStack app).  

## Important data source notes

- **Mathew et al. (2014) Geomorphology 228:307-319 is NOT a NER source** — that paper studies Garhwal Himalaya (Uttarakhand). Do not use it for NER calibration.
- Real positive labels come from: (a) the 9 documented real events in `historical_landslides` (is_synthetic=false), and (b) any COOLR or GSI Bhukosh CSV you load via `scripts/load_coolr_csv.sql`.
- Real sources for NER: NESAC/NERDRR NER Landslide Information System, NRSC/ISRO Landslide Atlas of India (1998-2022), GSI Bhukosh district reports.

## Prerequisites

```bash
pip install pandas numpy scikit-learn matplotlib seaborn psycopg2-binary python-dotenv scipy
```

Set `DATABASE_URL` in `.env` (same as Supabase connection string in your Lovable project settings).

## 0. Setup & imports

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from math import radians, cos, sin, asin, sqrt, atan2
from datetime import date, timedelta
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve, auc, classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
assert DATABASE_URL, 'Set DATABASE_URL in .env (Supabase connection string)'
print('✓ DATABASE_URL found')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load data from Supabase

In [ ]:
conn = psycopg2.connect(DATABASE_URL)

zones_df = pd.read_sql('''
    SELECT id, zone_name, state, district,
           centroid_lat, centroid_lng,
           mean_slope_deg, threshold_e_mm,
           threshold_i_coefficient, threshold_i_exponent,
           slope_source
    FROM risk_zones
    ORDER BY id
''', conn)

slides_df = pd.read_sql('''
    SELECT zone_id, event_date, severity, is_synthetic, source, lat, lng, hazard_type
    FROM historical_landslides
    ORDER BY event_date
''', conn)
slides_df['event_date'] = pd.to_datetime(slides_df['event_date'])

weather_df = pd.read_sql('''
    SELECT zone_id, reading_time::date AS reading_date,
           SUM(rainfall_mm) AS rainfall_mm,
           MAX(soil_moisture_pct) FILTER (WHERE soil_moisture_pct IS NOT NULL) AS soil_moisture_pct,
           source
    FROM weather_readings
    GROUP BY zone_id, reading_time::date, source
    ORDER BY zone_id, reading_date
''', conn)
weather_df['reading_date'] = pd.to_datetime(weather_df['reading_date'])

roads_df = pd.read_sql('SELECT zone_id, road_name, status, length_km FROM road_segments', conn)

conn.close()

real_slides = slides_df[~slides_df['is_synthetic']].copy()
synth_slides = slides_df[slides_df['is_synthetic']].copy()

# Step 4: Exclude GLOF-triggered events from the rainfall landslide training set
if 'hazard_type' in real_slides.columns:
    rainfall_real_slides = real_slides[real_slides['hazard_type'] == 'rainfall_slope_failure'].copy()
    glof_slides = real_slides[real_slides['hazard_type'] == 'glof_triggered'].copy()
else:
    rainfall_real_slides = real_slides.copy()
    glof_slides = pd.DataFrame()

print(f'Zones: {len(zones_df)}')
print(f'Landslide records total: {len(slides_df)}')
print(f'  Real total (is_synthetic=False): {len(real_slides)}')
print(f'    - Rainfall-triggered (for model calibration): {len(rainfall_real_slides)}')
print(f'    - GLOF-triggered (excluded from rainfall model): {len(glof_slides)}')
print(f'  Synthetic: {len(synth_slides)}')
print(f'Weather readings (daily, per zone): {len(weather_df)}')

if len(rainfall_real_slides) == 0:
    raise ValueError(
        '\n\nNo real landslide records found (is_synthetic=False). '
        'This notebook cannot produce valid metrics without real data. '
        'See docs/DATA_SOURCES.md for how to load real events from COOLR or GSI Bhukosh.'
    )

USING_SYNTHETIC = False
positives_raw = rainfall_real_slides.copy()
print(f'\n✓ Using {len(positives_raw)} real rainfall-triggered events for training.')

## 2. Exploratory Data Analysis

Answer before writing features (workflow doc Step 4):
1. How many real events do we have for NER?
2. What is their seasonal/monthly distribution?
3. Are they spatially clustered (affects split strategy)?
4. Does rainfall on event days visually separate from non-event days?

In [ ]:
print('=== Event distribution by state ===')
merged = positives_raw.merge(zones_df[['id','state','zone_name']], left_on='zone_id', right_on='id')
print(merged.groupby('state').size().sort_values(ascending=False).to_string())

print('\n=== Monthly distribution (should peak Jun-Sep) ===')
merged['month'] = merged['event_date'].dt.month
print(merged.groupby('month').size().to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Monthly distribution
merged.groupby('month').size().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Events by month (should peak Jun-Sep)'); axes[0].set_xlabel('Month')

# Events by zone
merged.groupby('zone_name').size().sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Events by zone')

# Severity distribution
merged['severity'].value_counts().plot(kind='pie', ax=axes[2], autopct='%1.0f%%')
axes[2].set_title('Severity distribution')

plt.tight_layout()
plt.savefig('docs/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✓ EDA plots saved to docs/eda_overview.png')

## 3. Rainfall feature engineering

In [ ]:
def build_rainfall_features(zone_id: int, as_of_date, weather_df: pd.DataFrame,
                             i_coef: float, i_exp: float, e_thr: float) -> dict:
    """
    Build rainfall features for a zone on a given date.
    Uses weather_readings data up to (but not including) the event date
    to avoid data leakage.

    Features (workflow doc Step 5):
    - rain_1/3/7/15/30d: cumulative rainfall in mm
    - rain_intensity_max_1d: peak single-day value in 30d window
    - antecedent_wetness_index (AWI): Σ rain_d × 0.9^(days_ago)
      where days_ago=0 is the most recent day
    - threshold_exceedance_flag: 1 if 3-day intensity > zone I-D threshold
    """
    zone_wx = weather_df[
        (weather_df['zone_id'] == zone_id) &
        (weather_df['reading_date'] < as_of_date)
    ].sort_values('reading_date').set_index('reading_date')

    if zone_wx.empty:
        return None  # Cannot compute features without weather data

    as_of = pd.Timestamp(as_of_date)

    def cumulative_rain(days: int) -> float:
        start = as_of - pd.Timedelta(days=days)
        mask = zone_wx.index >= start
        return float(zone_wx.loc[mask, 'rainfall_mm'].sum())

    def max_daily_rain(days: int) -> float:
        start = as_of - pd.Timedelta(days=days)
        mask = zone_wx.index >= start
        return float(zone_wx.loc[mask, 'rainfall_mm'].max() or 0.0)

    r_30d_series = zone_wx.loc[zone_wx.index >= as_of - pd.Timedelta(days=30), 'rainfall_mm']
    n = len(r_30d_series)
    decay = np.array([0.9 ** i for i in range(n)][::-1])
    awi = float((r_30d_series.values * decay).sum()) if n > 0 else 0.0

    r_3d = cumulative_rain(3)
    i_thr_3d = i_coef * (3.0 ** i_exp)  # I-D threshold for 3-day duration
    exceedance = 1 if (r_3d / 3.0) > i_thr_3d else 0

    return {
        'rain_1d': cumulative_rain(1),
        'rain_3d': r_3d,
        'rain_7d': cumulative_rain(7),
        'rain_15d': cumulative_rain(15),
        'rain_30d': cumulative_rain(30),
        'rain_intensity_max_1d': max_daily_rain(30),
        'antecedent_wetness_index': awi,
        'threshold_exceedance_flag': exceedance,
        'rain_3d_vs_e_thr': r_3d / e_thr if e_thr > 0 else 0.0,
    }

## 4. Soil moisture feature engineering

In [ ]:
def build_soil_features(zone_id: int, as_of_date, weather_df: pd.DataFrame) -> dict:
    """
    soil_moisture_latest: most recent value before event date (normalized 0-1).
    soil_moisture_7d_trend: relative change over past 7 days.
    Data comes from 'OM-SM-{zone_id}' station rows (Task B ingestion).
    Falls back to 0.5 (neutral) if no soil moisture data exists.
    """
    sm_rows = weather_df[
        (weather_df['zone_id'] == zone_id) &
        (weather_df['reading_date'] < as_of_date) &
        (weather_df['soil_moisture_pct'].notna())
    ].sort_values('reading_date')

    if sm_rows.empty:
        return {'soil_moisture_latest': 0.5, 'soil_moisture_7d_trend': 0.0,
                'soil_moisture_source': 'missing_fallback'}

    latest = sm_rows['soil_moisture_pct'].iloc[-1] / 100.0

    as_of = pd.Timestamp(as_of_date)
    week_ago_rows = sm_rows[sm_rows['reading_date'] >= as_of - pd.Timedelta(days=7)]
    if len(week_ago_rows) >= 2:
        oldest_val = week_ago_rows['soil_moisture_pct'].iloc[0] / 100.0
        trend = (latest - oldest_val) / max(oldest_val, 0.01)
        trend = float(np.clip(trend, -1.0, 1.0))
    else:
        trend = 0.0

    source = sm_rows['source'].iloc[-1] if 'source' in sm_rows.columns else 'unknown'
    return {
        'soil_moisture_latest': float(latest),
        'soil_moisture_7d_trend': trend,
        'soil_moisture_source': source,
    }

## 5. Terrain & proximity features

In [ ]:
def haversine_km(lat1, lng1, lat2, lng2) -> float:
    """Distance in km between two lat/lng points."""
    R = 6371.0
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlam = radians(lng2 - lng1)
    a = sin(dphi / 2)**2 + cos(phi1) * cos(phi2) * sin(dlam / 2)**2
    return R * 2 * asin(sqrt(a))


def build_terrain_features(zone_row) -> dict:
    """
    slope_norm: mean_slope_deg / 45 (0-1 scale)
    slope_sin: sin(slope_rad) — proportional to gravitational driving force
    slope_class: 0=Low(<15°), 1=Moderate(15-30°), 2=Steep(>30°)
    """
    slope_deg = float(zone_row['mean_slope_deg'])
    slope_rad = radians(slope_deg)
    return {
        'slope_norm': min(slope_deg / 45.0, 1.0),
        'slope_sin': float(sin(slope_rad)),
        'slope_class': 0 if slope_deg < 15 else (1 if slope_deg < 30 else 2),
    }


def build_proximity_features(zone_row, real_slides_df: pd.DataFrame) -> dict:
    """
    dist_to_nearest_event_km: Haversine distance from zone centroid to nearest
      real historical landslide point (is_synthetic=False).
    historical_event_density: count of real events within 50km radius / 4
      (capped at 1.0).
    """
    clat, clng = float(zone_row['centroid_lat']), float(zone_row['centroid_lng'])

    # Only use real events with valid coordinates
    located = real_slides_df.dropna(subset=['lat', 'lng'])

    if located.empty:
        return {'dist_to_nearest_event_km': 999.0, 'historical_event_density': 0.0}

    dists = located.apply(
        lambda r: haversine_km(clat, clng, float(r['lat']), float(r['lng'])), axis=1
    )
    nearest = float(dists.min())
    within_50 = int((dists <= 50.0).sum())

    return {
        'dist_to_nearest_event_km': nearest,
        'historical_event_density': min(within_50 / 4.0, 1.0),
    }


def build_temporal_features(event_date) -> dict:
    """
    day_of_year_sin / _cos: circular encoding (monsoon seasonality).
    is_monsoon: 1 if June–September, else 0.
    """
    doy = event_date.timetuple().tm_yday
    return {
        'day_of_year_sin': float(sin(2 * 3.14159 * doy / 365)),
        'day_of_year_cos': float(cos(2 * 3.14159 * doy / 365)),
        'is_monsoon': 1 if 6 <= event_date.month <= 9 else 0,
    }

## 6. Pseudo-absence sampling

Workflow doc Step 6:
- **Buffer**: 1 km around each real positive (zones with `mean_slope_deg > 5°` only — all 15 qualify)
- **Ratio**: 1:3 positive to negative
- **Temporal**: absence dates must not be within 14 days of a known event in the same zone

In [ ]:
PSEUDO_ABSENCE_BUFFER_KM = 1.0   # documented
PSEUDO_ABSENCE_SLOPE_MIN = 5.0   # degrees — documented
NEGATIVE_TO_POSITIVE_RATIO = 3   # documented
TEMPORAL_EXCLUSION_DAYS = 14     # documented

print(f'Pseudo-absence parameters:')
print(f'  Buffer: {PSEUDO_ABSENCE_BUFFER_KM} km around known positives')
print(f'  Slope minimum: {PSEUDO_ABSENCE_SLOPE_MIN}°')
print(f'  Negative:positive ratio: {NEGATIVE_TO_POSITIVE_RATIO}:1')
print(f'  Temporal exclusion: ±{TEMPORAL_EXCLUSION_DAYS} days of known event')

# Eligible zones (slope > 5° — all 15 in NER qualify)
eligible_zones = zones_df[zones_df['mean_slope_deg'] > PSEUDO_ABSENCE_SLOPE_MIN]
print(f'  Eligible zones: {len(eligible_zones)}/{len(zones_df)}')

rng = np.random.default_rng(RANDOM_SEED)
negatives = []

# Year range: same as real events ± 2 years
min_year = int(positives_raw['event_date'].dt.year.min()) - 2
max_year = int(positives_raw['event_date'].dt.year.max())
year_pool = list(range(max(min_year, 2010), max_year + 1))

n_negatives_needed = len(positives_raw) * NEGATIVE_TO_POSITIVE_RATIO

attempts = 0
while len(negatives) < n_negatives_needed and attempts < n_negatives_needed * 20:
    attempts += 1
    # Pick a random eligible zone
    z_row = eligible_zones.sample(1, random_state=rng.integers(0, 99999)).iloc[0]
    zone_id = int(z_row['id'])

    # Pick a random date in the year pool
    y = int(rng.choice(year_pool))
    m = int(rng.integers(1, 13))
    d_max = 28 if m == 2 else (30 if m in [4,6,9,11] else 31)
    d = int(rng.integers(1, d_max + 1))
    try:
        candidate_date = pd.Timestamp(year=y, month=m, day=d)
    except Exception:
        continue

    # Temporal exclusion: not within TEMPORAL_EXCLUSION_DAYS of a known event in this zone
    zone_events = positives_raw[positives_raw['zone_id'] == zone_id]['event_date']
    too_close = any(
        abs((candidate_date - evt).days) <= TEMPORAL_EXCLUSION_DAYS
        for evt in zone_events
    )
    if too_close:
        continue

    # Spatial check: zone centroid must be > PSEUDO_ABSENCE_BUFFER_KM from any positive
    clat, clng = float(z_row['centroid_lat']), float(z_row['centroid_lng'])
    positives_located = positives_raw.dropna(subset=['lat','lng'])
    too_close_spatial = any(
        haversine_km(clat, clng, float(r['lat']), float(r['lng'])) < PSEUDO_ABSENCE_BUFFER_KM
        for _, r in positives_located.iterrows()
    )
    if too_close_spatial:
        continue

    negatives.append({'zone_id': zone_id, 'event_date': candidate_date, 'label': 0})

neg_df = pd.DataFrame(negatives)
print(f'\n✓ Sampled {len(neg_df)} pseudo-absences (target: {n_negatives_needed})')
if len(neg_df) < n_negatives_needed:
    print(f'  ⚠ Only {len(neg_df)} / {n_negatives_needed} sampled — consider relaxing buffer or temporal window')

## 7. Build the full feature matrix

In [ ]:
def build_row(zone_id, event_date, label, zones_df, weather_df, real_slides_df):
    z = zones_df[zones_df['id'] == zone_id].iloc[0]

    rain_feats = build_rainfall_features(
        zone_id, event_date, weather_df,
        i_coef=float(z['threshold_i_coefficient']),
        i_exp=float(z['threshold_i_exponent']),
        e_thr=float(z['threshold_e_mm']),
    )
    if rain_feats is None:
        return None  # Skip rows with no weather data

    soil_feats = build_soil_features(zone_id, event_date, weather_df)
    terrain_feats = build_terrain_features(z)
    prox_feats = build_proximity_features(z, real_slides_df)
    temp_feats = build_temporal_features(event_date)

    row = {
        'zone_id': zone_id,
        'event_date': event_date,
        'district': z['district'],
        'state': z['state'],
        'label': label,
        **rain_feats,
        **{k: v for k, v in soil_feats.items() if k != 'soil_moisture_source'},
        **terrain_feats,
        **prox_feats,
        **temp_feats,
    }
    return row


# Build positive rows
pos_rows = []
for _, slide in positives_raw.iterrows():
    row = build_row(slide['zone_id'], slide['event_date'], 1,
                    zones_df, weather_df, real_slides)
    if row is not None:
        pos_rows.append(row)

# Build negative rows
neg_rows = []
for _, neg in neg_df.iterrows():
    row = build_row(neg['zone_id'], neg['event_date'], 0,
                    zones_df, weather_df, real_slides)
    if row is not None:
        neg_rows.append(row)

feature_df = pd.DataFrame(pos_rows + neg_rows).reset_index(drop=True)

print(f'Feature matrix shape: {feature_df.shape}')
print(f'  Positives: {(feature_df["label"]==1).sum()}')
print(f'  Negatives: {(feature_df["label"]==0).sum()}')

## 8. Completeness assertion — NO silent NaN

**This cell must pass before training. Any NaN at this point means a feature engineering bug.**

In [ ]:
FEATURE_COLS = [
    'rain_1d', 'rain_3d', 'rain_7d', 'rain_15d', 'rain_30d',
    'rain_intensity_max_1d', 'antecedent_wetness_index', 'threshold_exceedance_flag',
    'rain_3d_vs_e_thr',
    'soil_moisture_latest', 'soil_moisture_7d_trend',
    'slope_norm', 'slope_sin', 'slope_class',
    'dist_to_nearest_event_km', 'historical_event_density',
    'day_of_year_sin', 'day_of_year_cos', 'is_monsoon',
]

X = feature_df[FEATURE_COLS]
y = feature_df['label'].values
groups = feature_df['district'].values

nan_counts = X.isnull().sum()
if nan_counts.sum() > 0:
    print('NaN counts per feature:')
    print(nan_counts[nan_counts > 0])
    raise AssertionError(
        f'Feature matrix has {nan_counts.sum()} NaN values. '
        f'Fix feature engineering before training. '
        f'If weather data is missing for some event dates, the event rows '
        f'were already excluded in build_row() — check that pos_rows / neg_rows '
        f'are not None-filtered incorrectly.'
    )

print(f'✓ NaN assertion passed — {X.shape[0]} rows × {X.shape[1]} features, 0 NaN values')
print('\nFeature summary:')
X.describe().round(3)

## 9. Baseline model — published threshold only

Workflow doc Step 8: Run the published formula first. This is the floor. If ML cannot beat it, report that honestly.

In [ ]:
def threshold_only_predict(feature_df, zones_df):
    """
    Predict using only the published threshold exceedance flag.
    Returns binary predictions (1 = at-risk, 0 = not).
    """
    return feature_df['threshold_exceedance_flag'].values


y_thresh = threshold_only_predict(feature_df, zones_df)

from sklearn.metrics import precision_score, recall_score, f1_score

thresh_precision = precision_score(y, y_thresh, zero_division=0)
thresh_recall = recall_score(y, y_thresh, zero_division=0)
thresh_f1 = f1_score(y, y_thresh, zero_division=0)

# PR-AUC for threshold baseline (binary, so single point)
thresh_pr_precision = [thresh_precision, 1.0]
thresh_pr_recall = [thresh_recall, 0.0]
thresh_pr_auc = auc(thresh_pr_recall[::-1], thresh_pr_precision[::-1])

THRESHOLD_BASELINE_RESULTS = {
    'model': 'Threshold-only baseline (published NE-Himalaya / Sikkim I-D formula)',
    'pr_auc': thresh_pr_auc,
    'precision_at_threshold': thresh_precision,
    'recall_at_threshold': thresh_recall,
    'f1_at_threshold': thresh_f1,
    'recall_at_80_precision': thresh_recall if thresh_precision >= 0.80 else 0.0,
    'notes': 'Binary exceedance flag only. No training.',
}

print('Threshold-only baseline results:')
for k, v in THRESHOLD_BASELINE_RESULTS.items():
    print(f'  {k}: {v}')

print('\n' + classification_report(y, y_thresh, target_names=['No event', 'Event']))

## 10. Spatial cross-validation + model training

Workflow doc Step 7+9: GroupKFold by district (NOT random split).

In [ ]:
# Scale features for logistic regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_districts = len(set(groups))
n_splits = min(5, n_districts)  # Can't have more splits than districts
gkf = GroupKFold(n_splits=n_splits)

print(f'Spatial GroupKFold: {n_splits} splits by district (districts: {n_districts})')
print('⚠ GroupKFold ensures no district appears in both train and test — prevents spatial autocorrelation leakage.\n')

candidates = {
    'logistic_regression': LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=RANDOM_SEED),
    'random_forest': RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                             max_depth=5, random_state=RANDOM_SEED),
}

model_results = {}

for model_name, model in candidates.items():
    X_input = X_scaled if model_name == 'logistic_regression' else X.values
    proba = cross_val_predict(model, X_input, y, groups=groups, cv=gkf, method='predict_proba')[:, 1]

    precision, recall, thresholds = precision_recall_curve(y, proba)
    pr_auc_val = auc(recall, precision)

    idx_80 = next((i for i, p in enumerate(precision) if p >= 0.80), None)
    recall_at_80p = float(recall[idx_80]) if idx_80 is not None else 0.0

    model_results[model_name] = {
        'pr_auc': pr_auc_val,
        'recall_at_80_precision': recall_at_80p,
        'precision': precision,
        'recall': recall,
        'proba': proba,
    }

    print(f'{model_name}:')
    print(f'  PR-AUC: {pr_auc_val:.4f}')
    print(f'  Recall @ 80% precision: {recall_at_80p:.4f}')
    print()

## 11. Results table — threshold baseline vs. ML models

In [ ]:
rows_table = [
    {
        'Model': 'v0.1-hand-tuned (0.35/0.20/0.20/0.15/0.10)',
        'PR-AUC': '— (hand-tuned, not evaluated)',
        'Recall @ 80% precision': '— ',
        'Trained on': 'N/A',
        'Split': 'N/A',
    },
    {
        'Model': 'Threshold-only baseline (published I-D formula)',
        'PR-AUC': f"{THRESHOLD_BASELINE_RESULTS['pr_auc']:.4f}",
        'Recall @ 80% precision': f"{THRESHOLD_BASELINE_RESULTS['recall_at_80_precision']:.4f}",
        'Trained on': 'N/A (no training)',
        'Split': 'N/A',
    },
]

for mname, mres in model_results.items():
    rows_table.append({
        'Model': mname,
        'PR-AUC': f"{mres['pr_auc']:.4f}",
        'Recall @ 80% precision': f"{mres['recall_at_80_precision']:.4f}",
        'Trained on': f'{len(positives_raw)} real NER events',
        'Split': f'GroupKFold n={n_splits} by district',
    })

results_table = pd.DataFrame(rows_table)
print('=== Results: Threshold baseline vs. ML models ===')
print(results_table.to_string(index=False))

# Save for docs
results_table.to_csv('docs/model_evaluation_results.csv', index=False)
print('\n✓ Saved to docs/model_evaluation_results.csv')

## 12. PR curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'logistic_regression': 'steelblue', 'random_forest': 'coral'}

for mname, mres in model_results.items():
    ax.plot(mres['recall'], mres['precision'],
            color=colors.get(mname, 'gray'), lw=2,
            label=f"{mname} (PR-AUC={mres['pr_auc']:.3f})")

# Threshold baseline as a single point
ax.scatter([THRESHOLD_BASELINE_RESULTS['recall_at_threshold']],
           [THRESHOLD_BASELINE_RESULTS['precision_at_threshold']],
           color='green', s=100, zorder=5,
           label=f"Threshold baseline (P={THRESHOLD_BASELINE_RESULTS['precision_at_threshold']:.2f}, "
                 f"R={THRESHOLD_BASELINE_RESULTS['recall_at_threshold']:.2f})")

ax.axhline(0.80, color='orange', linestyle='--', label='80% precision line')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Landslide Risk Model — Precision-Recall Curves\n(Spatial GroupKFold CV by district)')
ax.legend(fontsize=8); ax.set_ylim(0, 1.05); ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig('docs/pr_curve.png', dpi=150)
plt.show()
print('✓ Saved docs/pr_curve.png')

## 13. Choose best model and extract coefficients

In [ ]:
lr_auc = model_results['logistic_regression']['pr_auc']
rf_auc = model_results['random_forest']['pr_auc']

# Decision: use LR unless RF beats it by > 5 percentage points
if rf_auc > lr_auc + 0.05:
    WINNING_MODEL_NAME = 'random_forest'
    print(f'✓ Random Forest wins (PR-AUC {rf_auc:.4f} vs LR {lr_auc:.4f}, gap > 0.05)')
else:
    WINNING_MODEL_NAME = 'logistic_regression'
    print(f'✓ Logistic Regression preferred (interpretable; PR-AUC {lr_auc:.4f}, RF={rf_auc:.4f})')

# Train final model on ALL data to extract coefficients
final_model = candidates[WINNING_MODEL_NAME]
X_final = X_scaled if WINNING_MODEL_NAME == 'logistic_regression' else X.values
final_model.fit(X_final, y)

# Extract weights
if WINNING_MODEL_NAME == 'logistic_regression':
    coefs = final_model.coef_[0]
    coefs_clipped = np.maximum(coefs, 0)  # risk factors are non-negative by design
    total = coefs_clipped.sum()
    coefs_norm = coefs_clipped / total if total > 0 else np.ones(len(FEATURE_COLS)) / len(FEATURE_COLS)

    print('\nLogistic regression coefficients (raw → non-neg normalized):')
    for feat, raw, norm in zip(FEATURE_COLS, coefs, coefs_norm):
        print(f'  {feat:<35}: raw={raw:+.4f}  norm={norm:.4f}')
else:
    importances = final_model.feature_importances_
    coefs_norm = importances / importances.sum()
    print('\nRandom Forest feature importances (normalized):')
    for feat, imp in zip(FEATURE_COLS, coefs_norm):
        print(f'  {feat:<35}: {imp:.4f}')

# Map to risk_model_config columns (5 core weights)
# weight_intensity ← rain_3d and rain_intensity_max_1d (take max)
# weight_antecedent ← rain_30d and antecedent_wetness_index (take max)
# weight_soil_moisture ← soil_moisture_latest
# weight_slope ← slope_norm and slope_sin (take max)
# weight_history ← historical_event_density
feat_idx = {f: i for i, f in enumerate(FEATURE_COLS)}

w_intensity  = max(coefs_norm[feat_idx['rain_3d']], coefs_norm[feat_idx['rain_intensity_max_1d']])
w_antecedent = max(coefs_norm[feat_idx['rain_30d']], coefs_norm[feat_idx['antecedent_wetness_index']])
w_soil       = coefs_norm[feat_idx['soil_moisture_latest']]
w_slope      = max(coefs_norm[feat_idx['slope_norm']], coefs_norm[feat_idx['slope_sin']])
w_history    = coefs_norm[feat_idx['historical_event_density']]

# Re-normalize to sum = 1
total_w = w_intensity + w_antecedent + w_soil + w_slope + w_history
w_intensity /= total_w; w_antecedent /= total_w; w_soil /= total_w
w_slope /= total_w; w_history /= total_w

WIN_PR_AUC = model_results[WINNING_MODEL_NAME]['pr_auc']
WIN_RECALL_AT_80P = model_results[WINNING_MODEL_NAME]['recall_at_80_precision']

print(f'\nMapped risk_model_config weights (sum={w_intensity+w_antecedent+w_soil+w_slope+w_history:.4f}):')
print(f'  weight_intensity:     {w_intensity:.4f}')
print(f'  weight_antecedent:    {w_antecedent:.4f}')
print(f'  weight_soil_moisture: {w_soil:.4f}')
print(f'  weight_slope:         {w_slope:.4f}')
print(f'  weight_history:       {w_history:.4f}')
print(f'  PR-AUC:               {WIN_PR_AUC:.4f}')
print(f'  Recall@80p:           {WIN_RECALL_AT_80P:.4f}')

## 14. Write results to risk_model_config

**Only proceed if you're satisfied with the results above.**
This cell updates the live database and triggers `recompute_risk()`.

In [ ]:
# Compute cutoffs from training data score distribution
def compute_score(row):
    f_intensity  = min(row['rain_3d'] / (float(zones_df[zones_df['id']==row['zone_id']]['threshold_i_coefficient'].iloc[0]) * (3.0 ** float(zones_df[zones_df['id']==row['zone_id']]['threshold_i_exponent'].iloc[0])) * 3), 1.0)
    f_antecedent = min(row['rain_30d'] / max(float(zones_df[zones_df['id']==row['zone_id']]['threshold_e_mm'].iloc[0]), 1), 1.0)
    f_soil       = row['soil_moisture_latest']
    f_slope      = row['slope_norm']
    f_history    = row['historical_event_density']
    return (w_intensity*f_intensity + w_antecedent*f_antecedent + w_soil*f_soil + w_slope*f_slope + w_history*f_history) * 100

pos_scores = feature_df[feature_df['label']==1].apply(compute_score, axis=1)
cutoff_moderate = max(35.0, round(float(np.percentile(pos_scores, 25)), 1))
cutoff_high     = max(cutoff_moderate + 8, round(float(np.percentile(pos_scores, 55)), 1))
cutoff_severe   = max(cutoff_high + 8,  round(float(np.percentile(pos_scores, 80)), 1))

print(f'Computed cutoffs — Moderate: {cutoff_moderate}, High: {cutoff_high}, Severe: {cutoff_severe}')

NEW_VERSION = f'v0.2-{WINNING_MODEL_NAME.replace("_","-")}-trained'
NOTES = (
    f'{WINNING_MODEL_NAME} trained on {len(positives_raw)} real NER landslide events '
    f'(NESAC/NERDRR, published literature). '
    f'Spatial GroupKFold CV n={n_splits} by district. '
    f'Pseudo-absence: {PSEUDO_ABSENCE_BUFFER_KM}km buffer, slope>{PSEUDO_ABSENCE_SLOPE_MIN}deg, '
    f'{NEGATIVE_TO_POSITIVE_RATIO}:1 neg:pos ratio, {TEMPORAL_EXCLUSION_DAYS}d exclusion. '
    f'NER-specific sources: NESAC NER Landslide Info System; NRSC/ISRO Landslide Atlas 1998-2022; '
    f'GSI Bhukosh district reports. Mathew et al. 2014 NOT used (covers Garhwal, not NER).'
)

CONFIRM = input(f'\nAbout to write {NEW_VERSION} to risk_model_config and run recompute_risk().\nType "yes" to proceed: ')
if CONFIRM.strip().lower() != 'yes':
    print('Aborted.')
else:
    conn2 = psycopg2.connect(DATABASE_URL)
    cur = conn2.cursor()
    cur.execute('UPDATE risk_model_config SET is_active = false WHERE is_active = true')
    cur.execute('''
        INSERT INTO risk_model_config (
          model_version, weight_intensity, weight_antecedent, weight_soil_moisture,
          weight_slope, weight_history,
          cutoff_moderate, cutoff_high, cutoff_severe,
          pr_auc, recall_at_80_precision, notes, is_active
        ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,true)
    ''', (NEW_VERSION,
          round(w_intensity,4), round(w_antecedent,4), round(w_soil,4),
          round(w_slope,4), round(w_history,4),
          cutoff_moderate, cutoff_high, cutoff_severe,
          round(WIN_PR_AUC,4), round(WIN_RECALL_AT_80P,4), NOTES))
    conn2.commit()
    cur.execute('SELECT recompute_risk()')
    conn2.commit()
    cur.close(); conn2.close()
    print(f'\n✓ {NEW_VERSION} written to risk_model_config.')
    print(f'  PR-AUC:       {WIN_PR_AUC:.4f}')
    print(f'  Recall@80p:   {WIN_RECALL_AT_80P:.4f}')
    print('  recompute_risk() called — scores updated.')